# 1. Inspect dataset


In [1]:
! echo "Downloading the Oxford hand dataset ..."
! wget http://www.robots.ox.ac.uk/~vgg/data/hands/downloads/hand_dataset.tar.gz
! echo "Download completed."

! tar -xf hand_dataset.tar.gz
! ln -s hand_dataset/evaluation_code/VOC2007/VOCdevkit VOCdevkit

--2025-12-26 11:41:46--  http://www.robots.ox.ac.uk/~vgg/data/hands/downloads/hand_dataset.tar.gz
Resolving www.robots.ox.ac.uk (www.robots.ox.ac.uk)... 129.67.94.2
Connecting to www.robots.ox.ac.uk (www.robots.ox.ac.uk)|129.67.94.2|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://www.robots.ox.ac.uk/~vgg/data/hands/downloads/hand_dataset.tar.gz [following]
--2025-12-26 11:41:47--  https://www.robots.ox.ac.uk/~vgg/data/hands/downloads/hand_dataset.tar.gz
Connecting to www.robots.ox.ac.uk (www.robots.ox.ac.uk)|129.67.94.2|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://thor.robots.ox.ac.uk/hands/hand_dataset.tar.gz [following]
--2025-12-26 11:41:47--  https://thor.robots.ox.ac.uk/hands/hand_dataset.tar.gz
Resolving thor.robots.ox.ac.uk (thor.robots.ox.ac.uk)... 129.67.95.98
Connecting to thor.robots.ox.ac.uk (thor.robots.ox.ac.uk)|129.67.95.98|:443... connected.
HTTP request sent, awaitin

In [2]:
from pathlib import Path
import collections

OXFORD_ROOT = "/content/hand_dataset"  # CHANGE THIS

root = Path(OXFORD_ROOT)
print("Root exists:", root.exists())
print("Top-level items:")
for p in sorted(root.iterdir()):
    print(" -", p.name)

# Try to locate validation_data/images and validation_data/annotations
val_candidates = list(root.rglob("validation_data"))
print("\nFound validation_data folders:")
for p in val_candidates[:20]:
    print(" -", p)

# Count extensions in annotations folder(s)
ann_folders = list(root.rglob("annotations"))
print("\nFound annotations folders:", len(ann_folders))
for ann_dir in ann_folders:
    exts = collections.Counter([f.suffix.lower() for f in ann_dir.iterdir() if f.is_file()])
    print("\nAnnotations dir:", ann_dir)
    print("Extension counts:", dict(exts))


Root exists: True
Top-level items:
 - .DS_Store
 - ._.DS_Store
 - ._README.txt
 - README.txt
 - evaluation_code
 - test_dataset
 - training_dataset
 - validation_dataset

Found validation_data folders:
 - /content/hand_dataset/validation_dataset/validation_data

Found annotations folders: 3

Annotations dir: /content/hand_dataset/test_dataset/test_data/annotations
Extension counts: {'.mat': 821, '.ds_store': 1, '': 1}

Annotations dir: /content/hand_dataset/validation_dataset/validation_data/annotations
Extension counts: {'.mat': 738, '.ds_store': 1, '': 1}

Annotations dir: /content/hand_dataset/training_dataset/training_data/annotations
Extension counts: {'.mat': 4069, '.ds_store': 1, '': 1}


In [3]:
from pathlib import Path
import random

# Pick ONE annotations folder that looks like validation_data/annotations
ann_dirs = [p for p in Path(OXFORD_ROOT).rglob("annotations")]
print("Annotation dirs found:", len(ann_dirs))
for i, d in enumerate(ann_dirs[:10]):
    print(i, d)

ann_dir = ann_dirs[0]  # change index if needed
img_dir = ann_dir.parent / "images"
print("\nUsing ann_dir:", ann_dir)
print("Guessed img_dir:", img_dir, "exists:", img_dir.exists())

ann_files = [f for f in ann_dir.iterdir() if f.is_file()]
img_files = []
if img_dir.exists():
    img_files = [f for f in img_dir.iterdir() if f.is_file()]

print("\nSample annotation files:")
for f in random.sample(ann_files, k=min(10, len(ann_files))):
    print(" -", f.name)

print("\nSample image files:")
for f in random.sample(img_files, k=min(10, len(img_files))):
    print(" -", f.name)


Annotation dirs found: 3
0 /content/hand_dataset/test_dataset/test_data/annotations
1 /content/hand_dataset/validation_dataset/validation_data/annotations
2 /content/hand_dataset/training_dataset/training_data/annotations

Using ann_dir: /content/hand_dataset/test_dataset/test_data/annotations
Guessed img_dir: /content/hand_dataset/test_dataset/test_data/images exists: True

Sample annotation files:
 - VOC2007_562.mat
 - VOC2007_603.mat
 - VOC2007_297.mat
 - VOC2007_55.mat
 - VOC2007_329.mat
 - VOC2010_52.mat
 - VOC2007_191.mat
 - VOC2007_147.mat
 - VOC2007_334.mat
 - VOC2007_416.mat

Sample image files:
 - VOC2007_163.jpg
 - VOC2007_63.jpg
 - VOC2007_560.jpg
 - VOC2007_628.jpg
 - VOC2007_561.jpg
 - VOC2007_632.jpg
 - VOC2007_316.jpg
 - VOC2010_90.jpg
 - VOC2007_150.jpg
 - VOC2007_328.jpg


In [4]:
from pathlib import Path
import random

import scipy.io as sio  # if this errors, tell me and I’ll adapt

ann_dir = [p for p in Path(OXFORD_ROOT).rglob("annotations")][0]  # adjust
mat_files = [f for f in ann_dir.iterdir() if f.is_file() and f.suffix.lower()==".mat"]
print("MAT files:", len(mat_files))

if mat_files:
    p = random.choice(mat_files)
    print("Inspecting:", p.name)
    data = sio.loadmat(str(p))
    keys = [k for k in data.keys() if not k.startswith("__")]
    print("Keys:", keys)
    for k in keys:
        v = data[k]
        try:
            print(" -", k, "type:", type(v), "shape:", getattr(v, "shape", None), "dtype:", getattr(v, "dtype", None))
        except Exception as e:
            print(" -", k, "error printing:", e)


MAT files: 821
Inspecting: VOC2010_112.mat
Keys: ['boxes']
 - boxes type: <class 'numpy.ndarray'> shape: (1, 2) dtype: object


In [5]:
import scipy.io as sio
import numpy as np
from pathlib import Path
import random

ann_dir = Path("/content/hand_dataset/training_dataset/training_data/annotations")  # change if needed
mat_path = random.choice(list(ann_dir.glob("*.mat")))
print("MAT:", mat_path.name)

data = sio.loadmat(str(mat_path))
boxes = data["boxes"]  # shape (1, N) object
print("boxes shape:", boxes.shape, "dtype:", boxes.dtype)

# how many boxes in this file?
N = boxes.shape[1]
print("N objects:", N)

# inspect first element
b0 = boxes[0, 0]
print("\nType of boxes[0,0]:", type(b0))

# If it's an ndarray, show shape and a preview
if isinstance(b0, np.ndarray):
    print("b0 ndarray shape:", b0.shape, "dtype:", b0.dtype)
    print("b0 preview:\n", b0)

# If it's a numpy void (MATLAB struct), show field names
if isinstance(b0, np.void):
    print("b0 is np.void (MATLAB struct). Fields:", b0.dtype.names)
    for name in b0.dtype.names:
        v = b0[name]
        print(f"  field {name}: type={type(v)}, shape={getattr(v,'shape',None)}, dtype={getattr(v,'dtype',None)}")
        # print small preview
        try:
            print("   preview:", v if np.size(v) < 20 else v.reshape(-1)[:20])
        except Exception as e:
            print("   preview error:", e)


MAT: Skin_7.mat
boxes shape: (1, 2) dtype: object
N objects: 2

Type of boxes[0,0]: <class 'numpy.ndarray'>
b0 ndarray shape: (1, 1) dtype: [('a', 'O'), ('b', 'O'), ('c', 'O'), ('d', 'O'), ('handtype', 'O'), ('truncated', 'O')]
b0 preview:
 [[(array([[167.45246585,  35.665999  ]]), array([[144.24928853,  49.47224985]]), array([[157.11478561,  71.09436966]]), array([[180.31796293,  57.28811882]]), array(['R'], dtype='<U1'), array([], shape=(0, 0), dtype=uint8))]]


In [6]:
import scipy.io as sio
import numpy as np
from pathlib import Path
import random
from collections import Counter

ann_dir = Path("/content/hand_dataset/training_dataset/training_data/annotations")
mat_path = random.choice(list(ann_dir.glob("*.mat")))
data = sio.loadmat(str(mat_path))
boxes = data["boxes"]
N = boxes.shape[1]

print("MAT:", mat_path.name, "| N:", N)

types = Counter()
shapes = Counter()
dtypes = Counter()
struct_fields = Counter()

for i in range(N):
    bi = boxes[0, i]
    types[type(bi).__name__] += 1
    if isinstance(bi, np.ndarray):
        shapes[str(bi.shape)] += 1
        dtypes[str(bi.dtype)] += 1
    if isinstance(bi, np.void):
        struct_fields[str(bi.dtype.names)] += 1

print("Types:", types)
print("Shapes (ndarray only):", shapes)
print("Dtypes (ndarray only):", dtypes)
if struct_fields:
    print("Struct fields:", struct_fields)


MAT: VOC2007_322.mat | N: 4
Types: Counter({'ndarray': 4})
Shapes (ndarray only): Counter({'(1, 1)': 4})
Dtypes (ndarray only): Counter({"[('a', 'O'), ('b', 'O'), ('c', 'O'), ('d', 'O'), ('handtype', 'O'), ('truncated', 'O')]": 4})


In [7]:
import scipy.io as sio
import numpy as np
from pathlib import Path
import random

ann_dir = Path("/content/hand_dataset/training_dataset/training_data/annotations")
mat_path = random.choice(list(ann_dir.glob("*.mat")))
data = sio.loadmat(str(mat_path))
boxes = data["boxes"]
N = boxes.shape[1]

print("MAT:", mat_path.name, "| N:", N)

for i in range(min(3, N)):
    bi = boxes[0, i]
    print("\n--- object", i, "---")
    print("type:", type(bi))
    if isinstance(bi, np.ndarray):
        print("shape:", bi.shape, "dtype:", bi.dtype)
        print(bi)
    elif isinstance(bi, np.void):
        print("fields:", bi.dtype.names)
        for name in bi.dtype.names:
            v = bi[name]
            print(" ", name, ":", v)


MAT: Inria_340.mat | N: 1

--- object 0 ---
type: <class 'numpy.ndarray'>
shape: (1, 1) dtype: [('a', 'O'), ('b', 'O'), ('c', 'O'), ('d', 'O'), ('handtype', 'O'), ('truncated', 'O')]
[[(array([[253.96275506, 369.96822396]]), array([[234.09606847, 379.41887282]]), array([[246.64508808, 405.79880113]]), array([[266.51177467, 396.34815227]]), array(['R'], dtype='<U1'), array([], shape=(0, 0), dtype=uint8))]]


# 2. Convert to yolo-obb format

In [8]:
# %%
import shutil
from pathlib import Path
import numpy as np
import cv2
import scipy.io as sio
import yaml

# ===== CONFIG =====
OXFORD_ROOT = "/content/hand_dataset"
OUT_ROOT    = "/content/hand_dataset_yolo_obb"

SUBSETS = {
    "train": ("training_dataset/training_data/images",    "training_dataset/training_data/annotations"),
    "val":   ("validation_dataset/validation_data/images","validation_dataset/validation_data/annotations"),
    "test":  ("test_dataset/test_data/images",            "test_dataset/test_data/annotations"),
}

IMG_EXTS = {".jpg",".jpeg",".png",".bmp"}

# Confirmed by you:
SWAP_XY = True    # stored as (y,x) -> convert to (x,y)
MINUS1  = False   # do NOT subtract 1

# Optional: 1 class only
USE_HANDTYPE_AS_CLASS = False
HANDTYPE_MAP = {"L": 0, "R": 1}
UNKNOWN_HANDTYPE_CLASS = 0

# Filtering controls
OOB_TOLERANCE_PX = 2.0   # allow slightly outside image before skipping
MIN_AREA_PX      = 20.0  # drop degenerate polygons
# ===================


def polygon_area_xy(poly_xy: np.ndarray) -> float:
    x = poly_xy[:, 0]
    y = poly_xy[:, 1]
    return 0.5 * float(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))

def reorder_clockwise(poly_xy: np.ndarray) -> np.ndarray:
    c = poly_xy.mean(axis=0)
    ang = np.arctan2(poly_xy[:, 1] - c[1], poly_xy[:, 0] - c[0])
    order = np.argsort(ang)
    poly = poly_xy[order]
    if polygon_area_xy(poly) > 0:
        poly = poly[::-1]
    return poly

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def extract_quads_from_mat(mat_path: Path):
    data = sio.loadmat(str(mat_path))
    if "boxes" not in data:
        return []
    boxes = data["boxes"]
    out = []
    for i in range(boxes.shape[1]):
        s = boxes[0,i][0,0]
        names = set(s.dtype.names or ())
        if not {"a","b","c","d"}.issubset(names):
            continue

        def pt(name):
            return np.array(s[name], dtype=np.float32).reshape(-1)[:2]

        quad = np.stack([pt("a"), pt("b"), pt("c"), pt("d")], axis=0)  # stored as (y,x)

        handtype = "U"
        if "handtype" in names:
            ht = s["handtype"]
            if isinstance(ht, np.ndarray) and ht.size > 0:
                handtype = str(ht.reshape(-1)[0])
            else:
                handtype = str(ht)

        out.append({"quad_raw": quad, "handtype": handtype})
    return out


def convert_split(split: str, img_rel: str, ann_rel: str):
    img_dir = Path(OXFORD_ROOT) / img_rel
    ann_dir = Path(OXFORD_ROOT) / ann_rel
    assert img_dir.exists(), f"Missing: {img_dir}"
    assert ann_dir.exists(), f"Missing: {ann_dir}"

    out_img_dir = Path(OUT_ROOT) / "images" / split
    out_lbl_dir = Path(OUT_ROOT) / "labels" / split
    ensure_dir(out_img_dir)
    ensure_dir(out_lbl_dir)

    img_paths = sorted([p for p in img_dir.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS])
    print(f"[{split}] images={len(img_paths)}")

    converted = 0
    skipped_no_labels = 0
    missing_mat = 0

    for img_path in img_paths:
        mat_path = ann_dir / f"{img_path.stem}.mat"
        if not mat_path.exists():
            missing_mat += 1
            continue

        im = cv2.imread(str(img_path))
        if im is None:
            continue
        h, w = im.shape[:2]

        objs = extract_quads_from_mat(mat_path)
        lines = []

        for obj in objs:
            q = obj["quad_raw"].astype(np.float32)

            # Apply the confirmed interpretation
            if MINUS1:
                q = q - 1.0
            if SWAP_XY:
                q = q[:, ::-1]   # (y,x) -> (x,y)

            # Skip if far outside image
            if (q[:,0].min() < -OOB_TOLERANCE_PX or q[:,0].max() > (w-1)+OOB_TOLERANCE_PX or
                q[:,1].min() < -OOB_TOLERANCE_PX or q[:,1].max() > (h-1)+OOB_TOLERANCE_PX):
                continue

            # Light clip in pixel space (safe)
            q[:,0] = np.clip(q[:,0], 0, w-1)
            q[:,1] = np.clip(q[:,1], 0, h-1)

            if abs(polygon_area_xy(q)) < MIN_AREA_PX:
                continue

            q = reorder_clockwise(q)

            # Normalize (NO clamp here!)
            qn = q.copy()
            qn[:,0] /= float(w)
            qn[:,1] /= float(h)

            if USE_HANDTYPE_AS_CLASS:
                cls = HANDTYPE_MAP.get(obj["handtype"], UNKNOWN_HANDTYPE_CLASS)
            else:
                cls = 0

            flat = qn.reshape(-1)
            lines.append(f"{cls} " + " ".join(f"{v:.6f}" for v in flat))

        if not lines:
            skipped_no_labels += 1
            continue

        shutil.copy2(str(img_path), str(out_img_dir / img_path.name))
        (out_lbl_dir / f"{img_path.stem}.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")
        converted += 1

    print(f"[{split}] converted={converted}, skipped_no_labels={skipped_no_labels}, missing_mat={missing_mat}")


# Run
Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)

for split, (img_rel, ann_rel) in SUBSETS.items():
    convert_split(split, img_rel, ann_rel)

# data.yaml
names = {0: "hand"} if not USE_HANDTYPE_AS_CLASS else {0:"left_hand", 1:"right_hand"}
data_yaml = {
    "path": str(Path(OUT_ROOT).resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": names,
}
(Path(OUT_ROOT) / "data.yaml").write_text(yaml.safe_dump(data_yaml, sort_keys=False), encoding="utf-8")
print("Wrote", Path(OUT_ROOT) / "data.yaml")


[train] images=4069
[train] converted=3953, skipped_no_labels=116, missing_mat=0
[val] images=738
[val] converted=738, skipped_no_labels=0, missing_mat=0
[test] images=821
[test] converted=794, skipped_no_labels=27, missing_mat=0
Wrote /content/hand_dataset_yolo_obb/data.yaml


In [9]:
# %%
from pathlib import Path
import numpy as np
import cv2
import random

YOLO_ROOT = "/content/hand_dataset_yolo_obb"

def load_lines(p: Path):
    txt = p.read_text(encoding="utf-8", errors="ignore").strip()
    return [ln.strip() for ln in txt.splitlines() if ln.strip()] if txt else []

def parse_line(line: str):
    parts = line.split()
    if len(parts) != 9:
        return None
    cls = int(float(parts[0]))
    poly = np.array(list(map(float, parts[1:])), dtype=np.float32).reshape(4,2)
    return cls, poly

def poly_area_norm(poly: np.ndarray) -> float:
    x = poly[:,0]; y = poly[:,1]
    return 0.5 * float(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))

def validate_split(split: str):
    img_dir = Path(YOLO_ROOT) / "images" / split
    lbl_dir = Path(YOLO_ROOT) / "labels" / split
    imgs = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in [".jpg",".jpeg",".png",".bmp"]])

    missing = 0
    bad = 0
    out_range = 0
    tiny = 0

    for img_path in imgs:
        lbl_path = lbl_dir / f"{img_path.stem}.txt"
        if not lbl_path.exists():
            missing += 1
            continue
        for ln in load_lines(lbl_path):
            parsed = parse_line(ln)
            if parsed is None:
                bad += 1
                continue
            _, poly = parsed
            if np.any(poly < 0) or np.any(poly > 1):
                out_range += 1
            if abs(poly_area_norm(poly)) < 1e-6:
                tiny += 1

    print(f"[{split}] images={len(imgs)} missing={missing} bad={bad} out_of_range={out_range} tiny={tiny}")

for s in ["train","val","test"]:
    validate_split(s)

def draw_poly(im, poly_px):
    poly_px = poly_px.astype(np.int32).reshape((-1,1,2))
    cv2.polylines(im, [poly_px], True, (0,255,0), 2)
    pts = poly_px.reshape(-1,2)
    for i,(x,y) in enumerate(pts):
        cv2.circle(im, (int(x),int(y)), 3, (0,0,255), -1)
        cv2.putText(im, str(i), (int(x)+4,int(y)+4), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,0,0), 1)
    return im

def visualize(split="val", n=60):
    img_dir = Path(YOLO_ROOT) / "images" / split
    lbl_dir = Path(YOLO_ROOT) / "labels" / split
    out_dir = Path(YOLO_ROOT) / "viz" / split
    out_dir.mkdir(parents=True, exist_ok=True)

    imgs = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in [".jpg",".jpeg",".png",".bmp"]])
    picks = random.sample(imgs, k=min(n, len(imgs)))

    for img_path in picks:
        im = cv2.imread(str(img_path))
        if im is None:
            continue
        h,w = im.shape[:2]
        lbl_path = lbl_dir / f"{img_path.stem}.txt"
        if not lbl_path.exists():
            continue

        for ln in load_lines(lbl_path):
            parsed = parse_line(ln)
            if parsed is None:
                continue
            _, poly = parsed
            poly_px = poly.copy()
            poly_px[:,0] *= w
            poly_px[:,1] *= h
            im = draw_poly(im, poly_px)

        cv2.imwrite(str(out_dir / img_path.name), im)

    print("Saved to:", out_dir)

visualize("val", 80)


[train] images=3953 missing=0 bad=0 out_of_range=0 tiny=0
[val] images=738 missing=0 bad=0 out_of_range=0 tiny=0
[test] images=794 missing=0 bad=0 out_of_range=0 tiny=0
Saved to: /content/hand_dataset_yolo_obb/viz/val


In [10]:
from pathlib import Path
import numpy as np
import cv2
import random

YOLO_ROOT = "/content/hand_dataset_yolo_obb"  # change if needed

def load_lines(p: Path):
    txt = p.read_text(encoding="utf-8", errors="ignore").strip()
    return [ln.strip() for ln in txt.splitlines() if ln.strip()] if txt else []

def parse_line(line: str):
    parts = line.split()
    if len(parts) != 9:
        return None
    cls = int(float(parts[0]))
    poly = np.array(list(map(float, parts[1:])), dtype=np.float32).reshape(4, 2)
    return cls, poly

def draw_poly_with_arrow(im, poly_px, arrow_scale=0.45):
    """
    poly_px: (4,2) float pixel coords
    arrow points from centroid toward midpoint of the longest edge (outward-ish).
    arrow_scale controls arrow length relative to max(w,h) of the polygon.
    """
    h_img, w_img = im.shape[:2]

    # Draw polygon
    pts = poly_px.astype(np.int32)
    cv2.polylines(im, [pts.reshape((-1,1,2))], True, (0,255,0), 2)

    # Draw corner points + indices
    for i, (x, y) in enumerate(pts):
        cv2.circle(im, (int(x), int(y)), 3, (0,0,255), -1)
        cv2.putText(im, str(i), (int(x)+4, int(y)+4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,0,0), 1)

    # Centroid
    c = poly_px.mean(axis=0)

    # Find longest edge (i -> i+1)
    edges = []
    for i in range(4):
        p1 = poly_px[i]
        p2 = poly_px[(i+1) % 4]
        length = float(np.linalg.norm(p2 - p1))
        mid = (p1 + p2) / 2.0
        edges.append((length, mid))

    edges.sort(key=lambda x: x[0], reverse=True)
    longest_len, mid_longest = edges[0]

    # Arrow direction: centroid -> midpoint of longest edge
    v = mid_longest - c
    norm = float(np.linalg.norm(v)) + 1e-9
    v = v / norm

    # Arrow length: relative to polygon size
    bbox_w = poly_px[:,0].max() - poly_px[:,0].min()
    bbox_h = poly_px[:,1].max() - poly_px[:,1].min()
    base = max(bbox_w, bbox_h)
    L = max(20.0, arrow_scale * base)

    start = (int(c[0]), int(c[1]))
    end = (int(c[0] + v[0] * L), int(c[1] + v[1] * L))

    # Draw arrow
    cv2.arrowedLine(im, start, end, (255,255,0), 2, tipLength=0.25)

    return im

def visualize_with_arrows(split="val", n=60):
    img_dir = Path(YOLO_ROOT) / "images" / split
    lbl_dir = Path(YOLO_ROOT) / "labels" / split
    out_dir = Path(YOLO_ROOT) / "viz_arrows" / split
    out_dir.mkdir(parents=True, exist_ok=True)

    imgs = sorted([p for p in img_dir.iterdir()
                   if p.is_file() and p.suffix.lower() in [".jpg",".jpeg",".png",".bmp"]])
    picks = random.sample(imgs, k=min(n, len(imgs)))

    for img_path in picks:
        im = cv2.imread(str(img_path))
        if im is None:
            continue
        h, w = im.shape[:2]

        lbl_path = lbl_dir / f"{img_path.stem}.txt"
        if not lbl_path.exists():
            continue

        for ln in load_lines(lbl_path):
            parsed = parse_line(ln)
            if parsed is None:
                continue
            cls, poly = parsed

            # denormalize to pixels
            poly_px = poly.copy()
            poly_px[:, 0] *= w
            poly_px[:, 1] *= h

            im = draw_poly_with_arrow(im, poly_px)

        cv2.imwrite(str(out_dir / img_path.name), im)

    print("Saved to:", out_dir)

# Run
visualize_with_arrows(split="val", n=80)


Saved to: /content/hand_dataset_yolo_obb/viz_arrows/val


In [11]:
# ! rm -rf /content/hand_dataset_yolo_obb

# 3. Training on YOLO11n-OBB

In [12]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 57.7 MB/s eta 0:00:00


In [13]:
import ultralytics
from ultralytics import YOLO
print("ultralytics version:", ultralytics.__version__)

# show if OBB task is available
try:
    from ultralytics.models.yolo.obb import OBBTrainer
    print("OBBTrainer: available ✅")
except Exception as e:
    print("OBBTrainer: not found ❌", repr(e))


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
ultralytics version: 8.3.241
OBBTrainer: available ✅


In [14]:
from ultralytics import YOLO

MODEL = "yolo11n-obb.pt"   # or "yolov11n-obb.yaml" if you have a config
try:
    m = YOLO(MODEL)
    print("Loaded model ✅", MODEL)
    print("Task:", m.task)
except Exception as e:
    print("Cannot load ❌", MODEL)
    print("Error:", repr(e))


Loaded model ✅ yolo11n-obb.pt
Task: obb


In [15]:
import os
from pathlib import Path
import json
import random
import cv2
import numpy as np

import ultralytics
from ultralytics import YOLO

print("Ultralytics:", ultralytics.__version__)

Ultralytics: 8.3.241


In [16]:
DATA_ROOT = Path("/content/hand_dataset_yolo_obb")  # <-- your YOLO-OBB dataset root
DATA_YAML = DATA_ROOT / "data.yaml"

MODEL_NAME = "yolo11n-obb.pt"  # Ultralytics will auto-download if missing

assert DATA_YAML.exists(), f"Missing data.yaml at: {DATA_YAML}"
print("Dataset YAML:", DATA_YAML)
print("Model:", MODEL_NAME)

Dataset YAML: /content/hand_dataset_yolo_obb/data.yaml
Model: yolo11n-obb.pt


In [17]:
required_dirs = [
    DATA_ROOT / "images" / "train",
    DATA_ROOT / "images" / "val",
    DATA_ROOT / "labels" / "train",
    DATA_ROOT / "labels" / "val",
]
for d in required_dirs:
    print(d, "exists:", d.exists())

# Count images/labels
def count_files(dirpath: Path, exts):
    if not dirpath.exists():
        return 0
    return sum(1 for p in dirpath.iterdir() if p.is_file() and p.suffix.lower() in exts)

img_exts = {".jpg",".jpeg",".png",".bmp"}
train_imgs = count_files(DATA_ROOT/"images/train", img_exts)
val_imgs   = count_files(DATA_ROOT/"images/val", img_exts)
train_lbls = count_files(DATA_ROOT/"labels/train", {".txt"})
val_lbls   = count_files(DATA_ROOT/"labels/val", {".txt"})

print("\nCounts:")
print(" train images:", train_imgs, " train labels:", train_lbls)
print(" val images  :", val_imgs,   " val labels  :", val_lbls)


/content/hand_dataset_yolo_obb/images/train exists: True
/content/hand_dataset_yolo_obb/images/val exists: True
/content/hand_dataset_yolo_obb/labels/train exists: True
/content/hand_dataset_yolo_obb/labels/val exists: True

Counts:
 train images: 3953  train labels: 3953
 val images  : 738  val labels  : 738


In [18]:
def load_lines(p: Path):
    txt = p.read_text(encoding="utf-8", errors="ignore").strip()
    return [ln.strip() for ln in txt.splitlines() if ln.strip()] if txt else []

def parse_line(line: str):
    parts = line.split()
    if len(parts) != 9:
        return None
    cls = int(float(parts[0]))
    poly = np.array(list(map(float, parts[1:])), dtype=np.float32).reshape(4,2)
    return cls, poly

def poly_area_norm(poly: np.ndarray) -> float:
    x = poly[:,0]; y = poly[:,1]
    return 0.5 * float(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))

def check_split(split="val", max_print=10, sample_n=300):
    img_dir = DATA_ROOT / "images" / split
    lbl_dir = DATA_ROOT / "labels" / split
    imgs = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in img_exts])
    if not imgs:
        print("No images in", img_dir)
        return

    picks = random.sample(imgs, k=min(sample_n, len(imgs)))

    bad = 0
    out_range = 0
    tiny = 0
    empty = 0
    shown = 0

    for img_path in picks:
        lbl_path = lbl_dir / f"{img_path.stem}.txt"
        if not lbl_path.exists():
            bad += 1
            if shown < max_print:
                print("Missing label:", img_path.name)
                shown += 1
            continue

        lines = load_lines(lbl_path)
        if not lines:
            empty += 1
            if shown < max_print:
                print("Empty label:", lbl_path.name)
                shown += 1
            continue

        for ln in lines:
            parsed = parse_line(ln)
            if parsed is None:
                bad += 1
                if shown < max_print:
                    print("Bad format:", lbl_path.name, "->", ln)
                    shown += 1
                continue
            _, poly = parsed
            if np.any(poly < 0) or np.any(poly > 1):
                out_range += 1
                if shown < max_print:
                    print("Out of range:", lbl_path.name, poly)
                    shown += 1
            if abs(poly_area_norm(poly)) < 1e-6:
                tiny += 1
                if shown < max_print:
                    print("Tiny area:", lbl_path.name, "area=", abs(poly_area_norm(poly)))
                    shown += 1

    print(f"\n[{split}] checked={len(picks)} images")
    print(" bad format/missing:", bad)
    print(" empty labels      :", empty)
    print(" out_of_range polys:", out_range)
    print(" tiny polys        :", tiny)

check_split("train", sample_n=500)
check_split("val", sample_n=300)



[train] checked=500 images
 bad format/missing: 0
 empty labels      : 0
 out_of_range polys: 0
 tiny polys        : 0

[val] checked=300 images
 bad format/missing: 0
 empty labels      : 0
 out_of_range polys: 0
 tiny polys        : 0


In [19]:
def draw_poly(im, poly_px):
    poly_px = poly_px.astype(np.int32).reshape((-1,1,2))
    cv2.polylines(im, [poly_px], True, (0,255,0), 2)
    pts = poly_px.reshape(-1,2)
    for i,(x,y) in enumerate(pts):
        cv2.circle(im, (int(x),int(y)), 3, (0,0,255), -1)
        cv2.putText(im, str(i), (int(x)+4,int(y)+4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,0,0), 1)
    return im

def visualize(split="val", n=20, out_dir=None):
    img_dir = DATA_ROOT / "images" / split
    lbl_dir = DATA_ROOT / "labels" / split
    out_dir = Path(out_dir or (DATA_ROOT / "viz_traincheck" / split))
    out_dir.mkdir(parents=True, exist_ok=True)

    imgs = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in img_exts])
    picks = random.sample(imgs, k=min(n, len(imgs)))

    for img_path in picks:
        im = cv2.imread(str(img_path))
        if im is None:
            continue
        h,w = im.shape[:2]
        lbl_path = lbl_dir / f"{img_path.stem}.txt"
        if not lbl_path.exists():
            continue

        for ln in load_lines(lbl_path):
            parsed = parse_line(ln)
            if parsed is None:
                continue
            _, poly = parsed
            poly_px = poly.copy()
            poly_px[:,0] *= w
            poly_px[:,1] *= h
            im = draw_poly(im, poly_px)

        cv2.imwrite(str(out_dir / img_path.name), im)

    print("Saved viz to:", out_dir)

visualize("val", n=40)


Saved viz to: /content/hand_dataset_yolo_obb/viz_traincheck/val


In [20]:
model = YOLO(MODEL_NAME)
pre_val = model.val(
    data=str(DATA_YAML),
    imgsz=640,
    device=0,   # A100
)
print("Pretrained val done.")

Ultralytics 8.3.241 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
YOLO11n-obb summary (fused): 109 layers, 2,656,648 parameters, 0 gradients, 6.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 965.8±394.8 MB/s, size: 31.6 KB)
val: Scanning /content/hand_dataset_yolo_obb/labels/val... 738 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 738/738 1.3Kit/s 0.6s
val: New cache created: /content/hand_dataset_yolo_obb/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 47/47 8.6it/s 5.5s
                   all        738       1847    0.00694   0.000541    0.00349    0.00209
                 plane        738       1847    0.00694   0.000541    0.00349    0.00209
Speed: 0.5ms preprocess, 1.3ms inference, 0.0ms loss, 2.3ms postprocess per image
Results saved to /content/runs/obb/val
Pretrained val done.


In [21]:
results = model.train(
    data=str(DATA_YAML),
    imgsz=640,
    epochs=120,
    batch=64,
    device=0,
    workers=8,

    optimizer="SGD",
    lr0=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    cos_lr=True,
    warmup_epochs=3,

    # augmentations (safe for hands)
    degrees=10,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    fliplr=0.5,
    flipud=0.0,

    # bookkeeping
    name="oxford_hand_yolo11n_obb",
)

Ultralytics 8.3.241 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/hand_dataset_yolo_obb/data.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=120, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-obb.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=oxford_hand_yolo11n_obb, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patienc

# 4. Validate best.pt model

In [22]:
run_dir = Path(model.trainer.save_dir)  # where this run was saved
best_pt = run_dir / "weights" / "best.pt"
print("Run dir:", run_dir)
print("Best:", best_pt, "exists:", best_pt.exists())

best_model = YOLO(str(best_pt))
best_val = best_model.val(
    data=str(DATA_YAML),
    imgsz=640,
    device=0,
)
print("Best model val done.")

Run dir: /content/runs/obb/oxford_hand_yolo11n_obb
Best: /content/runs/obb/oxford_hand_yolo11n_obb/weights/best.pt exists: True
Ultralytics 8.3.241 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
YOLO11n-obb summary (fused): 109 layers, 2,653,918 parameters, 0 gradients, 6.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1268.3±559.5 MB/s, size: 30.1 KB)
val: Scanning /content/hand_dataset_yolo_obb/labels/val.cache... 738 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 738/738 1.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 47/47 7.7it/s 6.1s
                   all        738       1847      0.847      0.758      0.857       0.61
Speed: 0.5ms preprocess, 0.8ms inference, 0.0ms loss, 2.5ms postprocess per image
Results saved to /content/runs/obb/val2
Best model val done.


In [23]:
pred_out = best_model.predict(
    source=str(DATA_ROOT / "images" / "val"),
    imgsz=640,
    conf=0.25,
    iou=0.7,
    device=0,
    save=True,
    max_det=50,
    name="oxford_hand_yolo11n_obb_pred",
)
print("Prediction done. Check runs/obb/oxford_hand_yolo11n_obb_pred/")


image 1/738 /content/hand_dataset_yolo_obb/images/val/Movie_4weds_1.jpg: 512x640 3 hands, 94.5ms
image 2/738 /content/hand_dataset_yolo_obb/images/val/Movie_4weds_10.jpg: 512x640 4 hands, 9.7ms
image 3/738 /content/hand_dataset_yolo_obb/images/val/Movie_4weds_100.jpg: 512x640 3 hands, 9.5ms
image 4/738 /content/hand_dataset_yolo_obb/images/val/Movie_4weds_101.jpg: 512x640 3 hands, 9.5ms
image 5/738 /content/hand_dataset_yolo_obb/images/val/Movie_4weds_102.jpg: 512x640 3 hands, 9.6ms
image 6/738 /content/hand_dataset_yolo_obb/images/val/Movie_4weds_103.jpg: 512x640 3 hands, 9.5ms
image 7/738 /content/hand_dataset_yolo_obb/images/val/Movie_4weds_104.jpg: 512x640 3 hands, 9.8ms
image 8/738 /content/hand_dataset_yolo_obb/images/val/Movie_4weds_105.jpg: 512x640 4 hands, 9.5ms
image 9/738 /content/hand_dataset_yolo_obb/images/val/Movie_4weds_106.jpg: 512x640 3 hands, 9.4ms
image 10/738 /content/hand_dataset_yolo_obb/images/val/Movie_4weds_107.jpg: 512x640 3 hands, 9.4ms
image 11/738 /conten

In [24]:
!zip -rq yolo11n-obb-oxford.zip /content/runs

  adding: content/runs/ (stored 0%)
  adding: content/runs/obb/ (stored 0%)
  adding: content/runs/obb/val2/ (stored 0%)
  adding: content/runs/obb/val2/val_batch2_labels.jpg (deflated 14%)
  adding: content/runs/obb/val2/val_batch0_pred.jpg (deflated 16%)
  adding: content/runs/obb/val2/val_batch0_labels.jpg (deflated 17%)
  adding: content/runs/obb/val2/BoxF1_curve.png (deflated 18%)
  adding: content/runs/obb/val2/val_batch2_pred.jpg (deflated 14%)
  adding: content/runs/obb/val2/confusion_matrix_normalized.png (deflated 36%)
  adding: content/runs/obb/val2/BoxP_curve.png (deflated 15%)
  adding: content/runs/obb/val2/BoxR_curve.png (deflated 16%)
  adding: content/runs/obb/val2/confusion_matrix.png (deflated 36%)
  adding: content/runs/obb/val2/val_batch1_pred.jpg (deflated 16%)
  adding: content/runs/obb/val2/val_batch1_labels.jpg (deflated 17%)
  adding: content/runs/obb/val2/BoxPR_curve.png (deflated 19%)
  adding: content/runs/obb/oxford_hand_yolo11n_obb/ (stored 0%)
  adding: 